[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/01_error_analysis_and_floating_point/first_principles.ipynb)

# Topic 01: Error Analysis and Floating Point

## 1. First-Principles Intuition & Motivation

Every numerical algorithm lives in a world of finite information. A computer stores a real number with a finite number of bits, evaluates a function with a finite number of operations, and truncates every infinite process (limits, series, integrals) after finitely many steps. Numerical analysis begins by asking a single honest question: **how far is the number we computed from the number we wanted?**

Error analysis is the discipline that answers this question *before* the computation is run. It separates the two independent sources of error:

- **Truncation error** — the error of the *mathematical approximation itself*, present even in exact arithmetic (e.g., stopping a Taylor series after $k$ terms).
- **Rounding error** — the error introduced because *each arithmetic operation* is performed in finite precision.

A method is only as good as the balance between these two: refining a discretization reduces truncation error but multiplies the number of rounding-error-carrying operations.

### Why this matters for the rest of the curriculum

Every later topic in this module quotes an error bound: Newton's method converges quadratically *until rounding noise dominates*, central differences are $O(h^2)$ *until cancellation destroys them*, and normal equations square the condition number of a least-squares problem. None of those statements can be understood without the vocabulary built here: absolute and relative error, machine epsilon, the standard model of floating-point arithmetic, conditioning, and catastrophic cancellation.

> The sibling module `numerical_computing/` covers hands-on IEEE-754 practice, conditioning experiments and vectorization. Here we build the *analytical* theory of error that classical numerical analysis rests on.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (Absolute and relative error).** Let $x \in \mathbb{R}$ be an exact value and $\hat{x}$ an approximation. The *absolute error* and *relative error* are

$$
E_{\mathrm{abs}} = \lvert \hat{x} - x \rvert, \qquad E_{\mathrm{rel}} = \frac{\lvert \hat{x} - x \rvert}{\lvert x \rvert} \quad (x \neq 0).
$$

**Definition 2 (Significant digits).** $\hat{x}$ approximates $x$ to $p$ significant decimal digits if

$$
E_{\mathrm{rel}} \le \frac{1}{2} \times 10^{-p+1}.
$$

Relative error is the natural currency of floating-point computation because the spacing of representable numbers is *relative* to their magnitude.

**Definition 3 (Normalized floating-point system).** A floating-point system $F(\beta, p, e_{\min}, e_{\max})$ consists of numbers

$$
x = \pm \left( d_0 . d_1 d_2 \cdots d_{p-1} \right)_{\beta} \times \beta^{e}, \qquad d_0 \neq 0, \quad e_{\min} \le e \le e_{\max},
$$

with base $\beta$, precision $p$ and exponent range $[e_{\min}, e_{\max}]$. IEEE double precision is $F(2, 53, -1022, 1023)$.

**Definition 4 (Machine epsilon and unit roundoff).** Machine epsilon is the gap between $1$ and the next representable number, $\varepsilon_{\mathrm{mach}} = \beta^{1-p}$; the *unit roundoff* is $u = \frac{1}{2}\beta^{1-p}$. For double precision,

$$
\varepsilon_{\mathrm{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}, \qquad u = 2^{-53} \approx 1.11 \times 10^{-16}.
$$

**Theorem 1 (Rounding is relatively accurate).** For every real $x$ in the normalized range of $F$, the rounded value satisfies

$$
\mathrm{fl}(x) = x(1 + \delta), \qquad \lvert \delta \rvert \le u.
$$

*Proof sketch.* The representable numbers in the binade $[\beta^{e}, \beta^{e+1})$ are spaced $\beta^{e+1-p}$ apart, so round-to-nearest incurs absolute error at most $\tfrac{1}{2}\beta^{e+1-p}$; dividing by $\lvert x \rvert \ge \beta^{e}$ bounds the relative error by $\tfrac{1}{2}\beta^{1-p} = u$. $\square$

**Definition 5 (Standard model of arithmetic).** IEEE arithmetic guarantees that each basic operation $\circ \in \{+, -, \times, \div\}$ returns the correctly rounded exact result:

$$
\mathrm{fl}(a \circ b) = (a \circ b)(1 + \delta), \qquad \lvert \delta \rvert \le u.
$$

This single axiom is the foundation of all rounding-error analysis: *each operation is nearly exact; danger arises only from the accumulation and amplification of these tiny perturbations.*

**Definition 6 (Condition number of a scalar function).** For differentiable $f$ with $f(x) \neq 0$, the relative condition number at $x$ is

$$
\kappa_f(x) = \left\lvert \frac{x f'(x)}{f(x)} \right\rvert .
$$

It measures the factor by which relative input error is amplified into relative output error.

**Definition 7 (Forward and backward error).** Let $\hat{y}$ be the computed value of $y = f(x)$. The *forward error* is $\lvert \hat{y} - y \rvert$. A *backward error* is the smallest $\lvert \Delta x \rvert$ such that $\hat{y} = f(x + \Delta x)$ exactly. An algorithm is *backward stable* if it always returns the exact answer to a nearby problem, $\lvert \Delta x \rvert / \lvert x \rvert = O(u)$.

**Theorem 2 (Rule of thumb).** Forward relative error $\lesssim \kappa_f(x) \times$ backward relative error. Accuracy is a joint property of the *problem* (conditioning) and the *algorithm* (stability).

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — First-order error propagation through a function

**Claim.** If $\hat{x} = x(1 + \varepsilon)$ with $\lvert \varepsilon \rvert \ll 1$ and $f$ is twice continuously differentiable with $f(x) \neq 0$, then

$$
\frac{\lvert f(\hat{x}) - f(x) \rvert}{\lvert f(x) \rvert} = \kappa_f(x)\,\lvert \varepsilon \rvert + O(\varepsilon^2).
$$

**Proof.** Taylor's theorem with Lagrange remainder gives, for some $\xi$ between $x$ and $\hat{x}$,

$$
f(\hat{x}) = f(x) + f'(x)(\hat{x} - x) + \tfrac{1}{2} f''(\xi)(\hat{x} - x)^2 .
$$

Since $\hat{x} - x = x\varepsilon$, dividing by $f(x)$ yields

$$
\frac{f(\hat{x}) - f(x)}{f(x)} = \frac{x f'(x)}{f(x)}\,\varepsilon + \frac{f''(\xi) x^2}{2 f(x)}\,\varepsilon^2 .
$$

Taking absolute values and absorbing the second term into $O(\varepsilon^2)$ gives the claim, with the leading coefficient exactly $\kappa_f(x)$. $\blacksquare$

### Proof 2 — Catastrophic cancellation: subtraction is ill-conditioned

**Claim.** Let $a \neq b$ have the same sign, and suppose $\hat{a} = a(1+\varepsilon_a)$, $\hat{b} = b(1+\varepsilon_b)$ with $\lvert \varepsilon_a \rvert, \lvert \varepsilon_b \rvert \le \varepsilon$. Then the relative error of $\hat{a} - \hat{b}$ as an approximation of $a - b$ can be as large as

$$
\frac{\lvert a \rvert + \lvert b \rvert}{\lvert a - b \rvert}\, \varepsilon .
$$

**Proof.** Compute the error directly:

$$
(\hat{a} - \hat{b}) - (a - b) = a\varepsilon_a - b\varepsilon_b .
$$

Hence

$$
\frac{\lvert (\hat{a} - \hat{b}) - (a - b) \rvert}{\lvert a - b \rvert} \le \frac{\lvert a \rvert \lvert \varepsilon_a \rvert + \lvert b \rvert \lvert \varepsilon_b \rvert}{\lvert a - b \rvert} \le \frac{\lvert a \rvert + \lvert b \rvert}{\lvert a - b \rvert}\,\varepsilon,
$$

and the bound is attained when $\varepsilon_a = -\varepsilon_b = \pm\varepsilon$ with $a, b \gt 0$. When $a \approx b$ the amplification factor $(\lvert a \rvert + \lvert b \rvert)/\lvert a - b \rvert$ is huge: subtracting nearly equal numbers can annihilate *all* significant digits. Note carefully: the subtraction itself is done exactly (Theorem 1); the damage is the amplification of *pre-existing* errors in $a$ and $b$. $\blacksquare$

### Proof 3 — Digit-loss bound for cancellation

**Claim (loss-of-precision theorem).** If $a \gt b \gt 0$ and

$$
2^{-q} \le 1 - \frac{b}{a} \le 2^{-p},
$$

then the subtraction $a - b$ loses at least $p$ and at most $q$ binary significant digits.

**Proof.** The number of significant binary digits retained is governed by the ratio of the result's magnitude to the operands' magnitude. From the hypothesis,

$$
2^{-q} \le \frac{a - b}{a} \le 2^{-p},
$$

so $a - b$ is smaller than $a$ by a factor between $2^{-q}$ and $2^{-p}$. Writing $a - b = a \cdot 2^{-r}$ with $p \le r \le q$, normalizing the result shifts the significand left by $r$ bits, and the vacated low-order $r$ bits carry no information (they reflect only rounding noise in $a$ and $b$). Hence between $p$ and $q$ bits of precision are lost. $\blacksquare$

### Proof 4 — Forward error of recursive summation

**Claim.** Let $S_n = \sum_{i=1}^{n} x_i$ be computed by the recurrence $\hat{S}_1 = x_1$, $\hat{S}_k = \mathrm{fl}(\hat{S}_{k-1} + x_k)$. Then

$$
\lvert \hat{S}_n - S_n \rvert \le (n-1)\,u \sum_{i=1}^{n} \lvert x_i \rvert + O(u^2).
$$

**Proof.** By the standard model each addition satisfies $\hat{S}_k = (\hat{S}_{k-1} + x_k)(1 + \delta_k)$ with $\lvert \delta_k \rvert \le u$. Unrolling the recurrence,

$$
\hat{S}_n = \sum_{i=1}^{n} x_i \prod_{k=\max(i,2)}^{n} (1 + \delta_k).
$$

Each $x_i$ is multiplied by at most $n-1$ factors $(1+\delta_k)$. To first order, $\prod_k (1+\delta_k) = 1 + \sum_k \delta_k + O(u^2)$, so

$$
\hat{S}_n - S_n = \sum_{i=1}^{n} x_i \left( \sum_{k} \delta_k \right) + O(u^2), \qquad \left\lvert \sum_{k} \delta_k \right\rvert \le (n-1)u .
$$

Applying the triangle inequality gives the bound. Two lessons: (i) the error grows only linearly in $n$; (ii) if the $x_i$ have mixed signs and $\lvert S_n \rvert \ll \sum \lvert x_i \rvert$, the *relative* error blows up — summation inherits the conditioning of cancellation. Summing the smallest terms first, pairwise summation ($O(u \log n)$ growth) or Kahan compensated summation ($O(u)$ growth) sharpen the constant. $\blacksquare$

### Proof 5 — Sterbenz lemma: some subtractions are exact

**Claim.** In a binary floating-point system with subnormal numbers, if $a, b \gt 0$ are representable and

$$
\frac{b}{2} \le a \le 2b,
$$

then $a - b$ is exactly representable, so $\mathrm{fl}(a - b) = a - b$ with **zero** rounding error.

**Proof.** Assume without loss of generality $b \le a \le 2b$ (the case $a \le b$ is symmetric). Write $a = m_a \cdot 2^{e_a}$ and $b = m_b \cdot 2^{e_b}$ with integer significands $m_a, m_b \lt 2^{p}$. Since $b \le a \le 2b$, the exponents satisfy $e_b \le e_a \le e_b + 1$, so both numbers can be rescaled to the common exponent $e_b$: $a = m_a' \cdot 2^{e_b}$ with $m_a' = m_a 2^{e_a - e_b} \le 2m_a \lt 2^{p+1}$. Then

$$
a - b = (m_a' - m_b)\, 2^{e_b},
$$

where $m_a' - m_b$ is a non-negative integer. From $a \le 2b$ we get $a - b \le b$, hence $m_a' - m_b \le m_b \lt 2^{p}$: the difference fits in $p$ significand bits at exponent $e_b$, so it is representable (subnormals cover the case of a tiny exponent). A representable exact result is returned unchanged by correct rounding. $\blacksquare$

This is why the *rewritten*, cancellation-free forms of formulas work: the dangerous subtraction, performed on nearly equal operands, is itself exact — the cure is to make sure its operands are exact too.

## 4. Computational & Algorithmic Insights

### The truncation–rounding trade-off

Nearly every numerical method has a resolution parameter $h$ (step size, mesh width, series length). The total error typically splits as

$$
E(h) \approx \underbrace{C h^{q}}_{\text{truncation}} + \underbrace{\frac{c\, u}{h^{r}}}_{\text{rounding}},
$$

so $E$ traces a U-shaped curve in $h$: mathematics rewards small $h$, arithmetic punishes it. Minimizing gives an optimal $h^{*} \sim u^{1/(q+r)}$ — the reason central differences bottom out near $h \approx u^{1/3}$ (see Topic 05).

### Rewriting formulas to dodge cancellation

| Unstable form | Stable rewrite |
| :--- | :--- |
| $\sqrt{x^2 + 1} - x$ for large $x$ | $1 / (\sqrt{x^2 + 1} + x)$ |
| $1 - \cos x$ for small $x$ | $2 \sin^2(x/2)$ |
| $\log(1 + x)$ for tiny $x$ | `log1p(x)` |
| $e^{x} - 1$ for tiny $x$ | `expm1(x)` |
| quadratic root $\dfrac{-b + \sqrt{b^2 - 4ac}}{2a}$ when $b \gt 0$ | $\dfrac{2c}{-b - \sqrt{b^2 - 4ac}}$ |

### Algorithmic checklist

1. **Estimate the condition number first.** If $\kappa \approx 10^{k}$, expect to lose $k$ decimal digits no matter how the computation is organized ($\text{accurate digits} \approx 16 - \log_{10}\kappa$ in double precision).
2. **Prefer backward-stable algorithms.** They deliver the exact answer to a problem within distance $O(u)$ of yours — the best that finite data deserves.
3. **Never test floating-point equality.** Compare with a mixed tolerance $\lvert x - y \rvert \le \tau_{\mathrm{abs}} + \tau_{\mathrm{rel}} \max(\lvert x \rvert, \lvert y \rvert)$.
4. **Sum in the right order.** Ascending magnitude, pairwise, or compensated (Kahan) summation when $n$ is large or signs are mixed.
5. **Distrust subtractions of near-equal quantities** — hunt for an algebraically equivalent form (conjugates, half-angle identities, series expansions).

## 5. Real-World Physics & AI/ML Applications

**Deep learning in half precision.** Training in float16 ($u = 2^{-11} \approx 4.9 \times 10^{-4}$, smallest normal $\approx 6.1 \times 10^{-5}$) makes gradient underflow routine: small gradients round to zero and learning silently stalls. *Loss scaling* multiplies the loss by $2^{k}$ before backpropagation and divides the gradients afterward — a pure error-analysis fix. bfloat16 keeps float32's exponent range precisely to trade precision for dynamic range.

**The log-sum-exp trick.** Softmax and cross-entropy overflow for logits beyond $\approx 709$ in double precision. Because softmax is shift-invariant, subtracting $\max_i z_i$ from all logits makes every exponent $\le 0$ — an exact algebraic identity chosen purely for floating-point safety.

**Variance in one pass.** The textbook identity $\mathrm{Var}(x) = \mathbb{E}[x^2] - (\mathbb{E}[x])^2$ subtracts two large near-equal numbers when the mean dominates the spread; Welford's online update is the numerically sound alternative used by every serious statistics library.

**Physics: small differences of large energies.** Relativistic kinetic energy $E_k = (\gamma - 1) m c^2$ evaluated naively for $v \ll c$ cancels catastrophically because $\gamma \approx 1$; the series $\gamma - 1 = \tfrac{1}{2}\beta^2 + \tfrac{3}{8}\beta^4 + \cdots$ (with $\beta = v/c$) restores full accuracy. Orbital mechanics, structural analysis and quantum chemistry all routinely reformulate "large minus large" energy differences.

### Where each later topic leans on this one

| Later topic | Error-analysis dependence |
| :--- | :--- |
| Root finding (Topic 02) | Stopping criteria mix absolute/relative tolerances; achievable accuracy limited by $\kappa$ of the root |
| Interpolation (Topic 04) | Vandermonde ill-conditioning motivates Lagrange/Newton bases and Chebyshev nodes |
| Differentiation (Topic 05) | The $u^{1/2}$, $u^{1/3}$ optimal step sizes are direct corollaries of the U-curve |
| Quadrature (Topic 06) | Composite rules add thousands of terms — summation error analysis applies verbatim |
| Least squares (Topic 07) | $\kappa(A^{T} A) = \kappa(A)^2$ is the canonical conditioning catastrophe |
| ODE solvers (Topic 08) | Global error compounds per-step truncation and rounding error over many steps |

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Unit roundoff (double) | $u = 2^{-53} \approx 1.11 \times 10^{-16}$ |
| Standard model | $\mathrm{fl}(a \circ b) = (a \circ b)(1+\delta)$, $\lvert \delta \rvert \le u$ |
| Propagation | relative output error $\approx \kappa_f(x) \times$ relative input error |
| Cancellation | $\kappa_{-} = (\lvert a \rvert + \lvert b \rvert)/\lvert a - b \rvert$ |
| Recursive summation | error $\le (n-1)u \sum \lvert x_i \rvert + O(u^2)$ |
| Sterbenz | $b/2 \le a \le 2b \implies \mathrm{fl}(a-b) = a - b$ exactly |
| Digits heuristic | accurate digits $\approx 16 - \log_{10}\kappa$ |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Absolute/relative error, significant digits | Burden & Faires, *Numerical Analysis* | Ch. 1.2 |
| Floating-point systems, machine epsilon | Heath, *Scientific Computing* | Ch. 1.3 |
| Standard model $\mathrm{fl}(a \circ b) = (a \circ b)(1+\delta)$ | Higham, *Accuracy and Stability of Numerical Algorithms* | Ch. 2 |
| Conditioning and backward stability | Trefethen & Bau, *Numerical Linear Algebra* | Lectures 12–15 |
| Summation error analysis, Kahan summation | Higham, *Accuracy and Stability* | Ch. 4 |
| Sterbenz lemma, exact subtraction | Muller et al., *Handbook of Floating-Point Arithmetic* | Ch. 4 |
| Cancellation case studies | Goldberg, *What Every Computer Scientist Should Know About Floating-Point Arithmetic* | ACM Surveys 1991 |
| Conditioning of numerical problems | Quarteroni, Sacco & Saleri, *Numerical Mathematics* | Ch. 2 |

**Primary references.** Burden & Faires (Ch. 1); Heath (Ch. 1); Higham (Chs. 1–4); Trefethen & Bau (Lects. 12–15); Quarteroni et al. (Ch. 2); Goldberg (1991).